# AdRank-ML — end-to-end walkthrough

Generate a synthetic auction log, build 145 leakage-safe features, train and
calibrate CTR/CVR models, then run the GSP auction simulation. Everything reads
from `config/config.yaml` (the `demo` profile by default).

Run from the repo root with the project venv kernel.

In [ ]:
import sys, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, '../src')
from adrank.config import load_config
from adrank.utils.common import set_seed
cfg = load_config('../config/config.yaml')
cfg.scale['impressions'] = 500_000   # shrink for an interactive run
cfg.scale['n_users'] = 60_000
rng = set_seed(cfg.seed)
cfg.scale.profile, cfg.scale.impressions

In [ ]:
from adrank.data import generate as g
summary = g.generate(cfg, rng, chunk_size=250_000)
summary

In [ ]:
from adrank.features.engineering import build_features
feat_summary = build_features(cfg)
feat_summary

In [ ]:
from adrank.models.train import train
report = train(cfg)
report['ctr']['headline']

In [ ]:
from adrank.auction.gsp import simulate
sim = simulate(cfg, set_seed(cfg.seed + 7), n_auctions=20000)
sim['policies'], sim['adrank_vs_bid_only']

The reliability bins in `report['ctr']['headline']['_reliability_bins']` can be
plotted as a calibration curve (mean_pred vs mean_actual per bin).